# clock/13.1 — move-on intervention applied *mid-code*

The **move-on intervention** = transplant the mean *"moved-on"* signal (`mean_src`, the decision
heads' o_proj-input averaged over `sl->ml` completion sources at the pivot) into the **decision
heads** via `headpatched_generate`, strength `alpha`. Canonically it's applied **at line-end +
indent** (`… + "   "`, patch the pivot whitespace `-2`) — where the model is at a natural decision
point about whether to keep going or move on.

**Question:** what if we fire it *while the LM is mid-expression* (a few RHS tokens already written),
instead of at a clean line boundary?
- Does the LM **ignore** it (committed to finishing the expression)?
- How does **intervention strength** (`alpha`) change that?

Mechanism + head-set loaders copied from `09.1_goalhead_gaze`; the mid-code prompt reuses
`13_middle`'s `build_dissociation_prompt`. Qualitative `rich_print` only for now.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, random, logging, json
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from collections import defaultdict, Counter
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape
from src.utils import logging_utils, env_utils
logging_utils.configure(level=logging.INFO)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER = mt.n_layer
N_HEADS = mt.model.config.num_attention_heads
print(f"n_layer={N_LAYER}  n_embd={mt.n_embd}  n_heads={N_HEADS}")

2026-07-24 17:19:00 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-07-24 17:19:00 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-24 17:19:00 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-07-24 17:19:00 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-24 17:19:00 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-24 17:19:00 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-07-24 17:19:23 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-24 17:19:23 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-07-24 17:19:23 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64  n_embd=5120  n_heads=40


## 1. Decision heads + the move-on signal `mean_src`  (copied from 09.1 §1–§2)

In [3]:
from src.goals.head_loc_attn import HeadMask
from src.goals.oid_gaze import cache_root

VERSION              = "conditions"
COMPLETION_MASK_ROOT = os.path.join(cache_root(MODEL_NAME, VERSION), "completion_mask")
COMPLETION_MASK_TAG  = "lam10.0"
SEL_THRESHOLD        = 0.8

def _list_masks():
    return sorted([(t, os.path.join(COMPLETION_MASK_ROOT, t),
                    os.path.getmtime(os.path.join(COMPLETION_MASK_ROOT, t)))
                   for t in os.listdir(COMPLETION_MASK_ROOT)
                   if os.path.isdir(os.path.join(COMPLETION_MASK_ROOT, t))],
                  key=lambda x: x[2], reverse=True)
_m = [it for it in _list_masks() if COMPLETION_MASK_TAG in it[0]]
assert _m, f"no completion_mask under {COMPLETION_MASK_ROOT} with tag {COMPLETION_MASK_TAG!r}"
DCK = torch.load(os.path.join(_m[0][1], "mask.pt"), map_location="cpu", weights_only=False)
dec_layers = [int(x) for x in DCK["layers"]]
dmv = DCK["mask_values"].numpy()
decision_heads = [(dec_layers[li], int(h)) for li, h in np.argwhere(dmv > SEL_THRESHOLD)]
PATCH_POS = int(DCK["patch_tokens"][0]) if len(DCK.get("patch_tokens", [-2])) == 1 else -2
print(f"decision heads: {len(decision_heads)} (thr {SEL_THRESHOLD}) · "
      f"layers {min(dec_layers)}-{max(dec_layers)} · trained PATCH_POS={PATCH_POS}")

# reduced binary HeadMask over dec_layers (axis-0 aligned to mean_src / layers=dec_layers):
# forward_with_src_replacement indexes mask[li,h] / src[li] / layers[li] by the SAME li.
dec_mask = HeadMask(len(dec_layers), N_HEADS)
with torch.no_grad():
    dec_mask.values.zero_()
    for (l, h) in decision_heads:
        dec_mask.values[dec_layers.index(l), h] = 1.0

decision heads: 181 (thr 0.8) · layers 0-63 · trained PATCH_POS=-2


In [4]:
from src.goals.completion_pairs import build_completion_pairs
from src.goals.head_loc_attn import get_o_proj_inputs_batch

N_MEAN_SRC, MEAN_BATCH = 200, 16
src_pairs = build_completion_pairs(mt, n_pairs=N_MEAN_SRC, direction="sl->ml", same=True,
                                   patch_tokens=(PATCH_POS,), seed=1234, log=logger.info)
src_prompts = [p.source.sl_prompt for p in src_pairs]        # "moved-on" sources
acc = {l: [] for l in dec_layers}
for i in range(0, len(src_prompts), MEAN_BATCH):
    batch = src_prompts[i:i + MEAN_BATCH]
    cache = get_o_proj_inputs_batch(mt, batch, dec_layers, token_positions=[PATCH_POS])
    for l in dec_layers:
        acc[l].append(cache[l][:, 0, :].float().cpu())       # (b, n_embd)
mean_src = torch.stack([torch.cat(acc[l], 0).mean(0) for l in dec_layers])  # (n_dec_layers, n_embd)
print("mean move-on signal:", tuple(mean_src.shape))

2026-07-24 17:19:33 __main__ INFO       built 25/200 pairs (27 attempts)
2026-07-24 17:19:42 __main__ INFO       built 50/200 pairs (53 attempts)
2026-07-24 17:19:52 __main__ INFO       built 75/200 pairs (81 attempts)
2026-07-24 17:20:00 __main__ INFO       built 100/200 pairs (107 attempts)
2026-07-24 17:20:10 __main__ INFO       built 125/200 pairs (135 attempts)
2026-07-24 17:20:19 __main__ INFO       built 150/200 pairs (160 attempts)
2026-07-24 17:20:27 __main__ INFO       built 175/200 pairs (185 attempts)
2026-07-24 17:20:36 __main__ INFO       built 200/200 pairs (210 attempts)
2026-07-24 17:20:36 __main__ INFO     built 200 pairs from 210 attempts
mean move-on signal: (64, 5120)


## 2. Prompt builders + the move-on generator

Two application points for the *same* transplant:
- **control** — line end + `"   "` indent (the trained pivot), patch `-2`.
- **mid-code** — `build_dissociation_prompt` (cut at `= `) with `MID_DEPTH` RHS tokens already
  written, so the model is mid-expression; patch the last token `-1`.

In [5]:
from src.goals.multiline_prompts import build_example
from src.goals.condition_data import INT_PREDICATES

PIVOT_TRAILING_SPACE = False
_PRED_BY_ID = {p.id: p for p in INT_PREDICATES}

def build_dissociation_prompt(mt, rng, current_slot, n_conditions, naming="letters", lang="python"):
    ts = PIVOT_TRAILING_SPACE
    ex = build_example(mt=mt, rng=rng, target_slot=current_slot, n_conditions=n_conditions,
                       mode="chat", style="singleline", naming=naming, lang=lang,
                       prior_style="singleline")
    tvar = ex["target_var"]; fp = ex["full_prompt"]
    line_start = fp.rfind(f"\n    {tvar} =") + 1
    assign_eq  = fp.index("=", line_start)
    cut = assign_eq + 2 if ts else assign_eq + 1
    ex["dissoc_prompt"] = fp[:cut]; ex["current_slot"] = current_slot; ex["trailing_space"] = ts
    ids = mt.tokenizer(ex["dissoc_prompt"], add_special_tokens=False)["input_ids"]
    ex["prompt_ids"] = ids; ex["pivot_pos"] = len(ids) - 1
    return ex

def get_condition_references(mt, ex):
    ts = ex.get("trailing_space", PIVOT_TRAILING_SPACE)
    spec = ex["spec"]; out = []
    for k in range(ex["n_conditions"]):
        pid, prm, var = spec.predicate_ids[k], spec.predicate_params[k], ex["var_names"][k]
        pred = _PRED_BY_ID[pid]; has_sl = pred.has_singleline_for(ex["lang"])
        full_line = (pred.render_singleline(prm, var, lang=ex["lang"]) if has_sl
                     else " / ".join(pred.render_multiline(prm, var, lang=ex["lang"])))
        m = re.match(rf"^\s*{re.escape(var)}\s*=", full_line)
        rhs = full_line[m.end():] if m else full_line
        if ts and rhs.startswith(" "): rhs = rhs[1:]
        rhs_ids = mt.tokenizer(rhs, add_special_tokens=False)["input_ids"]
        out.append({"cond": k, "var": var, "pred": pid, "rhs": rhs, "rhs_ids": rhs_ids})
    return out

In [6]:
from src.tokens import prepare_input
from src.functional import generate_with_patch
from src.goals.goalscope import headpatched_generate
from src.goals.multiline_prompts import replace_in_code_newline_with_tabnewline

N_GEN = 64

def build_line_end(rng, target_slot, n_conditions):
    """Canonical control: model finished a line and typed the next-statement indent."""
    ex = build_example(mt=mt, rng=rng, target_slot=target_slot, n_conditions=n_conditions,
                       mode="chat", style="multiline", naming="letters")
    prompt = replace_in_code_newline_with_tabnewline(ex["full_prompt"]) + "   "
    return ex, prompt

def build_mid_code(rng, current_slot, n_conditions, depth_tokens):
    """Mid-expression: cut at `= `, then append the first `depth_tokens` of the canonical RHS."""
    ex = build_dissociation_prompt(mt, rng, current_slot=current_slot, n_conditions=n_conditions)
    refs = get_condition_references(mt, ex)
    rhs_ids = refs[current_slot]["rhs_ids"]
    prefix = mt.tokenizer.decode(rhs_ids[:depth_tokens])
    return ex, ex["dissoc_prompt"] + prefix, refs[current_slot]["rhs"]

def clean_generate(dest_tok, max_new=N_GEN):
    return generate_with_patch(mt, inputs=dest_tok, n_gen_per_prompt=1,
                               max_new_tokens=max_new, do_sample=False, remove_prefix=True)[0]

def move_on_generate(dest_tok, alpha, patch_pos, n_steer_tokens=1, max_new=N_GEN):
    """Transplant mean_src into the decision heads at `patch_pos` with strength `alpha`."""
    return headpatched_generate(mt, dest_inputs=dest_tok, src_o_proj_inps=mean_src,
                                mask=dec_mask, layers=dec_layers, max_new_tokens=max_new,
                                n_steer_tokens=n_steer_tokens, patch_input_positions=[patch_pos],
                                alpha=alpha)

## 3. Does the LM ignore move-on *mid-code*?  (clean vs patched, both points)

If the intervention "takes", the patched continuation abruptly **moves on** (newline / next var /
`return`). If the LM **ignores** it mid-expression, the patched continuation still finishes the RHS.

In [7]:
SEED       = 1123
N_COND     = 5
MID_SLOT   = 2
MID_DEPTH  = 3        # RHS tokens already written before we fire (knob)
ALPHA      = 2.0
CTRL_PATCH = PATCH_POS   # -2, the trained line-end pivot
MID_PATCH  = -1          # mid-expression: patch the current last token

# ---- control: line end + indent ----
ex_c, p_c = build_line_end(random.Random(SEED), target_slot=1, n_conditions=4)
tok_c = prepare_input(p_c, tokenizer=mt, padding_side="left")
rich_print("[bold underline]CONTROL — line end + indent (patch -2)[/bold underline]")
rich_print(f"[bold]clean         [/bold][white]{escape(p_c)}[/white][sea_green3]{escape(clean_generate(tok_c))}[/sea_green3]")
rich_print(f"[bold]move-on α={ALPHA} [/bold][white]{escape(p_c)}[/white][salmon1]{escape(move_on_generate(tok_c, ALPHA, CTRL_PATCH))}[/salmon1]")

# ---- mid-code ----
ex_m, p_m, rhs = build_mid_code(random.Random(SEED), current_slot=MID_SLOT, n_conditions=N_COND, depth_tokens=MID_DEPTH)
tok_m = prepare_input(p_m, tokenizer=mt, padding_side="left")
rich_print(f"\n[bold underline]MID-CODE — {MID_DEPTH} RHS tokens in (patch -1)[/bold underline]  "
           f"[grey58]canonical RHS: {escape(repr(rhs))}[/grey58]")
rich_print(f"[bold]clean         [/bold][white]{escape(p_m)}[/white][sea_green3]{escape(clean_generate(tok_m))}[/sea_green3]")
rich_print(f"[bold]move-on α={ALPHA} [/bold][white]{escape(p_m)}[/white][salmon1]{escape(move_on_generate(tok_m, ALPHA, MID_PATCH))}[/salmon1]")

CONTROL — line end + indent (patch -2)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


clean         <|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n has at least 4 digits
2. n has more than 1 ones in its binary representation
3. n has more than 6 divisors
4. n is even

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code>    
def check_conditions(n: int):    
    a = len(str(abs(n)))    
    a = a >= 4    
    b = bin(abs(n)).count("1")    
    b = b > 1    
    c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1

move-on α=2.0 <|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n has at least 4 digits
2. n has more than 1 ones in its binary representation
3. n has more than 6 divisors
4. n is even

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code>    
def check_conditions(n: int):    
    a = len(str(abs(n)))    
    a = a >= 4    
    b = bin(abs(n)).count("1")    
    c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1: {a}, Condition 2:

MID-CODE — 3 RHS tokens in (patch -1)  canonical RHS: ' sum(1 for d in range(1, abs(n) + 1) if n % d == 0) > 6'

clean         <|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n has at least 4 digits
2. n has more than 1 ones in its binary representation
3. n has more than 6 divisors
4. n is even
5. the sum of n's digits is greater than 15

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code>
def check_conditions(n: int):
    a = len(str(abs(n))) >= 4
    b = bin(abs(n)).count("1") > 1
    c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0) > 6
    d = abs(n) % 2 == 0
    e = sum(int(digit) for digit in str(abs(n))) > 15
    print(f

move-on α=2.0 <|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:

1. n has at least 4 digits
2. n has more than 1 ones in its binary representation
3. n has more than 6 divisors
4. n is even
5. the sum of n's digits is greater than 15

For each condition, in the order listed, assign the boolean result of the check to its own local variable, using 
single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all 
conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.
Do not include any comments in your code.<|im_end|>
<|im_start|>assistant
<code>
def check_conditions(n: int):
    a = len(str(abs(n))) >= 4
    b = bin(abs(n)).count("1") > 1
    c = sum(1
            for i in range(1, abs(n) + 1)
            if abs(n) % i == 0) > 6
    d = abs(n) % 2 == 0
    e = sum(int(digit) for digit in str(abs(n))) > 15

## 4. Effect of intervention strength (`alpha` sweep) at both points

In [ ]:
ALPHAS = [0.5, 1, 2, 4, 8, 16]

for label, (tok, patch, prompt) in {
    "CONTROL — line end + indent (patch -2)": (tok_c, CTRL_PATCH, p_c),
    f"MID-CODE — {MID_DEPTH} RHS tokens in (patch -1)": (tok_m, MID_PATCH, p_m),
}.items():
    rich_print(f"\n[bold underline]{label}[/bold underline]  [grey58]…{escape(repr(prompt[-50:]))}[/grey58]")
    rich_print(f"[bold]clean   [/bold][sea_green3]{escape(clean_generate(tok))}[/sea_green3]")
    for a in ALPHAS:
        rich_print(f"[bold]α={a:<5}[/bold][salmon1]{escape(move_on_generate(tok, a, patch))}[/salmon1]")

CONTROL — line end + indent (patch -2)  …' a >= 4    \n    b = bin(abs(n)).count("1")    \n   '

clean    b = b > 1    
    c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1

α=0.5   c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1: {a}, Condition 2:

α=1     c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1: {a}, Condition 2:

α=2     c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1: {a}, Condition 2:

α=4     c = sum(1 for i in range(1, abs(n) + 1) if abs(n) % i == 0)    
    c = c > 6    
    d = n % 2 == 0    
    print(f"Condition 1: {a}, Condition 2:

... (7 outputs left out)

α=4        d = n % 2 == 0
    e = sum(int(digit) for digit in str(abs(n))) > 15
    print(f"a: {a}, b: {b}, c: {c}, d: {d}, e: {e}, all: {a and

α=8        d = n % 2 == 0
    e = sum(int(digit) for digit in str(abs(n))) > 15
    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, Condition 4: {d}, Condition

α=16    c = len([i for i in range(1, abs(n) + 1) if abs(n) % i == 0]) > 6
    d = abs(n) % 2 == 0
    e = sum(int(digit) for digit in str(abs(n))) > 1

: 